# 04 Modeling

Train and tune models here.

In [5]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 5.9 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.2 MB 6.7 MB/s eta 0:00:01
   ---------------------- ----------------- 4.7/8.2 MB 7.2 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.2 MB 6.4 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.2 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 6.4 MB/s  0:00:01
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   - -------------------------------------- 1.3/36.6 MB 6.6 MB/s eta 0:00:06
   --- ------------------------------------ 2.9/36.6 MB 6.9 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/36.6 MB 6.8 MB/s eta 0:00:05
   ------ --------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


#  44 – Kiểm tra Overfitting và Data Leakage
 

# Xác định dữ liệu và mục tiêu dự đoán

In [1]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "games_cleaned.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Kích thước dữ liệu:", df.shape)
print("\nDanh sách các cột:")
print(df.columns.tolist())

print("\nNăm dòng dữ liệu đầu tiên:")
display(df.head())


Kích thước dữ liệu: (125855, 39)

Danh sách các cột:
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

Năm dòng dữ liệu đầu tiên:


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...


# Chọn mục tiêu dự đoán
Mục tiêu dự đoán	Ý nghĩa	Lưu ý
Peak CCU	Lượng người chơi đồng thời cao nhất	Phù hợp để thử dự đoán mức độ thu hút người chơi
Positive	Số lượt đánh giá tích cực	Có thể chịu ảnh hưởng lớn từ quy mô người chơi
Price	Giá game	Có thể thử dự đoán giá dựa trên các thuộc tính của game
# chọn peak CCU trước

# Kiểm tra dữ liệu trước khi xây dựng mô hình

In [3]:
# Kiểm tra các cột có thể dùng để dự đoán Peak CCU

target = "Peak CCU"

candidate_features = [
    "Price",
    "Required age",
    "Achievements",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
]

print("Mục tiêu dự đoán:", target)

print("\nKiểu dữ liệu của các cột:")
display(df[[target] + candidate_features].dtypes.to_frame("Data type"))

print("\nSố lượng giá trị thiếu:")
display(df[[target] + candidate_features].isna().sum().to_frame("Missing values"))

print("\nThống kê mô tả:")
display(df[[target] + candidate_features].describe().T)

Mục tiêu dự đoán: Peak CCU

Kiểu dữ liệu của các cột:


,Data type
Peak CCU,int64
Price,float64
Required age,int64
Achievements,float64
Metacritic score,bool
User score,object
Positive,int64
Negative,int64
Recommendations,int64
Average playtime forever,object



Số lượng giá trị thiếu:


,Missing values
Peak CCU,0
Price,0
Required age,0
Achievements,125815
Metacritic score,0
User score,121597
Positive,0
Negative,0
Recommendations,0
Average playtime forever,102545



Thống kê mô tả:


,count,mean,std,min,25%,50%,75%,max
Peak CCU,125855.0,53.187200,3681.082965,0.0,0.00,0.00,0.000,1013936.00
Price,125855.0,4.811174,12.489329,0.0,0.59,2.39,5.545,999.98
Required age,125855.0,0.165365,1.642666,0.0,0.00,0.00,0.000,21.00
Achievements,40.0,99.175000,0.675107,98.0,99.00,99.00,100.000,100.00
Positive,125855.0,0.023916,1.376811,0.0,0.00,0.00,0.000,100.00
Negative,125855.0,1018.182091,27727.824406,0.0,0.00,4.00,35.000,7642084.00
Recommendations,125855.0,17.922220,139.696425,0.0,0.00,2.00,19.000,9821.00
Average playtime two weeks,125855.0,202.756831,11072.217573,0.0,0.00,0.00,0.000,3429544.00
Median playtime forever,125855.0,13.507711,266.925759,0.0,0.00,0.00,0.000,20088.00
Median playtime two weeks,125855.0,169.180327,11057.251586,0.0,0.00,0.00,0.000,3429544.00


# tạo mô hình thử nghiệm để kiểm tra overfitting
Chia dữ liệu thành tập huấn luyện và tập kiểm tra.



Dùng log1p(Peak CCU) để giảm ảnh hưởng của các giá trị CCU cực lớn.



So sánh mô hình đơn giản với Random Forest.



So sánh sai số giữa train và test để phát hiện dấu hiệu overfitting.

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Chọn các đặc trưng cơ bản, tránh dùng số liệu đánh giá
features = [
    "Price",
    "Required age",
    "Windows",
    "Mac",
    "Linux"
]

target = "Peak CCU"

# Chuyển các cột đầu vào về dạng số nếu có thể
X = df[features].copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y = np.log1p(df[target])

# Chia train/test trước khi huấn luyện
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Baseline": make_pipeline(
        SimpleImputer(strategy="median"),
        DummyRegressor(strategy="median")
    ),
    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Model,Train MAE,Test MAE,Train R2,Test R2
0,Baseline,0.3250,0.3238,-0.0999,-0.1000
1,Random Forest,0.4767,0.4855,0.1515,0.1208


Cách đọc kết quả
- Train MAE thấp nhưng Test MAE cao hơn nhiều: mô hình có dấu hiệu overfitting.
- Train và Test MAE đều cao: mô hình có thể chưa học được quy luật hữu ích.
- Random Forest tốt hơn Baseline trên tập test: có dấu hiệu mô hình học được thông tin hữu ích từ các đặc trưng đã chọn.
- Test R² âm: mô hình dự đoán còn kém hơn một phương án đơn giản dự đoán bằng giá trị trung bình của tập test.

# kiểm tra lại đặc điểm dữ liệu
Kết quả này giúp xác định vì sao mô hình khó dự đoán: dữ liệu có thể tập trung quá nhiều ở mức CCU bằng 0, trong khi một số ít game có giá trị cực lớn.

In [7]:
print("Tỷ lệ Peak CCU bằng 0:")
print((df["Peak CCU"] == 0).mean() * 100, "%")

print("\nSố game có Peak CCU > 0:")
print((df["Peak CCU"] > 0).sum())

print("\nCác phân vị của Peak CCU:")
display(df["Peak CCU"].quantile([0, 0.5, 0.75, 0.9, 0.95, 0.99, 1]))

Tỷ lệ Peak CCU bằng 0:
84.3629573715784 %

Số game có Peak CCU > 0:
19680

Các phân vị của Peak CCU:


0.00          0.00
0.50          0.00
0.75          0.00
0.90          2.00
0.95          8.00
0.99        215.46
1.00    1013936.00
Name: Peak CCU, dtype: float64


Kết quả này xác nhận một vấn đề quan trọng: dữ liệu Peak CCU bị lệch rất mạnh về phía 0. Đây có thể là một trong những nguyên nhân khiến mô hình khó dự đoán chính xác lượng người chơi.

In [8]:
#Ý nghĩa nhãn: 0 là game có Peak CCU = 0; 1 là game có Peak CCU > 0.

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Chọn đặc trưng và tạo nhãn phân loại
features = ["Price", "Required age", "Windows", "Mac", "Linux"]

X = df[features].copy()
y = (df["Peak CCU"] > 0).astype(int)

# Chuyển dữ liệu đầu vào về dạng số
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Chia tập train và test, giữ tỷ lệ hai nhóm
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Xây dựng mô hình phân loại
model_stage1 = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

model_stage1.fit(X_train, y_train)

# Dự đoán trên train và test
train_pred = model_stage1.predict(X_train)
test_pred = model_stage1.predict(X_test)

# Đánh giá kết quả
print("Accuracy train:", round(accuracy_score(y_train, train_pred), 4))
print("Accuracy test:", round(accuracy_score(y_test, test_pred), 4))

print("\nF1-score test:", round(f1_score(y_test, test_pred), 4))

print("\nClassification report trên tập test:")
print(classification_report(
    y_test, test_pred,
    target_names=["Peak CCU = 0", "Peak CCU > 0"],
    zero_division=0
))

print("Confusion matrix trên tập test:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=["Thực tế = 0", "Thực tế > 0"],
    columns=["Dự đoán = 0", "Dự đoán > 0"]
))

Accuracy train: 0.6303
Accuracy test: 0.6261

F1-score test: 0.3668

Classification report trên tập test:
              precision    recall  f1-score   support

Peak CCU = 0       0.92      0.61      0.73     21235
Peak CCU > 0       0.25      0.69      0.37      3936

    accuracy                           0.63     25171
   macro avg       0.58      0.65      0.55     25171
weighted avg       0.81      0.63      0.68     25171

Confusion matrix trên tập test:


,Dự đoán = 0,Dự đoán > 0
Thực tế = 0,13034,8201
Thực tế > 0,1210,2726


# Kết quả giai đoạn 1 cho thấy mô hình phát hiện được một phần game có Peak CCU > 0, nhưng vẫn dự đoán nhầm khá nhiều.

# giai đoạn 2: dự đoán lượng Peak CCU của các game có CCU lớn hơn 0.
Lưu ý: Giai đoạn 2 chỉ đánh giá hồi quy trên các game thực sự có Peak CCU > 0. Kết quả này chưa đại diện cho hiệu quả toàn bộ mô hình hai giai đoạn, vì chưa tính các lỗi phân loại từ giai đoạn 1.

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

# Chọn các game có Peak CCU > 0
positive_mask = df["Peak CCU"] > 0

X_positive = df.loc[positive_mask, features].copy()
y_positive = np.log1p(df.loc[positive_mask, "Peak CCU"])

# Chuyển các đặc trưng về dạng số
for col in X_positive.columns:
    X_positive[col] = pd.to_numeric(X_positive[col], errors="coerce")

# Chia dữ liệu theo đúng các chỉ số train/test đã dùng ở giai đoạn 1
train_index = X_train.index.intersection(X_positive.index)
test_index = X_test.index.intersection(X_positive.index)

X_train_stage2 = X_positive.loc[train_index]
y_train_stage2 = y_positive.loc[train_index]

X_test_stage2 = X_positive.loc[test_index]
y_test_stage2 = y_positive.loc[test_index]

# Tạo và huấn luyện mô hình hồi quy
model_stage2 = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
)

model_stage2.fit(X_train_stage2, y_train_stage2)

# Dự đoán trên train và test
train_pred_stage2 = model_stage2.predict(X_train_stage2)
test_pred_stage2 = model_stage2.predict(X_test_stage2)

# Đánh giá trên thang log
print("Số game CCU > 0 trong train:", len(X_train_stage2))
print("Số game CCU > 0 trong test:", len(X_test_stage2))

print("\nTrain MAE:", round(
    mean_absolute_error(y_train_stage2, train_pred_stage2), 4
))
print("Test MAE:", round(
    mean_absolute_error(y_test_stage2, test_pred_stage2), 4
))
print("Train R2:", round(
    r2_score(y_train_stage2, train_pred_stage2), 4
))
print("Test R2:", round(
    r2_score(y_test_stage2, test_pred_stage2), 4
))

Số game CCU > 0 trong train: 15744
Số game CCU > 0 trong test: 3936

Train MAE: 1.1908
Test MAE: 1.2155
Train R2: 0.1828
Test R2: 0.1263


Đánh giá kết quả
Chỉ số	Train	Test
MAE (thang log)	1.1908	1.2155
R²	0.1828	0.1263

- Overfitting: MAE giữa train và test chỉ chênh 0.0247, R² chênh 0.0565. Khoảng cách chưa lớn, dù vẫn có dấu hiệu suy giảm hiệu quả trên dữ liệu chưa thấy.
- Khả năng dự đoán: Test R² = 0.1263, nghĩa là mô hình giải thích được khoảng 12,63% biến thiên của mục tiêu trên thang log.
- Sai số: MAE được tính trên log1p(Peak CCU), không phải số người chơi trực tiếp.

# kết hợp hai giai đoạn
Cách tính:
- Nếu giai đoạn 1 dự đoán CCU = 0, dự đoán cuối cùng là 0.
- Nếu giai đoạn 1 dự đoán CCU > 0, lấy dự đoán hồi quy của giai đoạn 2 rồi chuyển về thang CCU bằng expm1.
- So sánh dự đoán cuối cùng với Peak CCU thực tế trên toàn bộ tập test.
# Lưu ý: Đoạn code này giả định hai mô hình đã được huấn luyện bằng đúng các đặc trưng và cách chia dữ liệu hiện tại. Chỉ số MAE và R² cuối cùng được tính trên thang CCU gốc; do đó không so sánh trực tiếp MAE này với MAE trên thang log ở giai đoạn 2.

In [10]:
# Kết hợp dự đoán của hai giai đoạn trên tập test

stage1_pred = model_stage1.predict(X_test)

stage2_pred_log = model_stage2.predict(X_test)
stage2_pred_ccu = np.expm1(stage2_pred_log)

# Chỉ giữ dự đoán CCU dương nếu giai đoạn 1 phân loại là dương
final_pred = np.where(stage1_pred == 1, stage2_pred_ccu, 0)

# Giá trị CCU thực tế trên toàn bộ tập test
y_test_ccu = df.loc[X_test.index, "Peak CCU"].to_numpy()

print("MAE toàn bộ test:", round(
    mean_absolute_error(y_test_ccu, final_pred), 2
))
print("R2 toàn bộ test:", round(
    r2_score(y_test_ccu, final_pred), 4
))

MAE toàn bộ test: 37.77
R2 toàn bộ test: 0.017


Kết luận nhiệm vụ # 44 đến hiện tại
- Overfitting: Chưa thấy dấu hiệu rõ rệt qua chênh lệch train-test ở hai giai đoạn riêng lẻ.
- Data leakage: Đã hạn chế một số nguy cơ bằng cách không dùng các cột đánh giá và đề xuất, nhưng chưa thể khẳng định đã loại bỏ hoàn toàn leakage.
- Hiệu quả mô hình: Mô hình hai giai đoạn có R² toàn bộ test chỉ 0,017, nên khả năng dự đoán CCU thực tế còn yếu.
- Nguyên nhân cần xem xét: 84,36% game có CCU bằng 0, phân bố lệch mạnh và các đặc trưng hiện tại (Price, Required age, Windows, Mac, Linux) có thể chưa đủ thông tin để dự đoán mức độ phổ biến.